# Evaluating Amazon Mistral Lite 7B finetuned for Skeptic Justifications dataset

### Install all dependencies

In [ ]:
# !pip install  accelerate --progress-bar off
# !pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
# !pip install  peft --progress-bar off
# !pip install  bitsandbytes --progress-bar off
# !pip install git+https://github.com/huggingface/transformers
# !pip install  xformers==0.0.21
# !pip install git+https://github.com/huggingface/trl.git
# !pip install deepspeed==0.9.5
# !pip install wandb
# !pip install vllm bert_score rouge nltk

### Loading Required Libraries

Next, we will load the required libraries for fine-tuning a Large Language Model (LLM)

In [1]:
import nltk
import torch
import random
import pandas as pd
import numpy as np
from rouge import Rouge
from bert_score import score
from vllm import LLM, SamplingParams
from nltk.translate.meteor_score import meteor_score

nltk.download('wordnet')
nltk.download('omw-1.4')

rouge = Rouge()

[nltk_data] Downloading package wordnet to /home/ec2-user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ec2-user/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
torch_version = torch.__version__
if torch_version == "2.0.1+cu118":
    print(f"Torch version is satisfied: {torch.__version__}")
else:
    print("Torch version should be 2.0.1+cu118. Please ensure that before going further")

Torch version is satisfied: 2.0.1+cu118


### Loading the test set for Skeptic

In [3]:
df = pd.read_csv("data/claims_classify_labels_205_records.csv")

In [4]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,text,labels,predicted_label
0,7BObqdlgd5A,TCS Q2 Results 2023-24 Highlights | TCS Share ...,5paisa,TCS has just announced its Quarterly Results. ...,"Hi guys, Quarter 2 FY24 result season ki shirv...","""Hi guys, the Q2 FY24 result season has begun ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQcICA...,https://www.youtube.com/watch?v=7BObqdlgd5A,The financial influencer claims that TCS's Q2 ...,The claims made by the influencer are true if ...,<|prompter|>You are a Financial Contrarian Wri...,true,true
1,R_OryHP3Fcg,Israel-Hamas Conflict's Impact on India #shorts,5paisa,Gain insight into how the Israel-Hamas Conflic...,बिलियन डौलर का ट्रेटिंग रेलेशन्चिप खत्रे में ह...,The trading relationship between Israel and Ha...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=R_OryHP3Fcg,The financial influencer claims that the confl...,The claims made by the influencer are plausibl...,<|prompter|>You are a Financial Contrarian Wri...,false,true
2,qX3J9Tq0XYU,YOUTUBE SE INCOME || MY FIRST INCOME,Amrev,YOUTUBE SE INCOME || MY FIRST INCOME\r\n\r\n...,so yeah parents do say a lot but it's okay wh...,"""So yeah, parents do say a lot, but it's okay....",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=qX3J9Tq0XYU,No claims made,No claims to analyse,<|prompter|>You are a Financial Contrarian Wri...,neutral,neutral
3,uQc6Tib329A,Growpital Review - 16% TAX FREE Return | Fixed...,Shrija Saha,Fixed Income - 16% Tax FREE Return - Growpita...,Fixed deposit with 16% returns वो भी tax-free ...,"""Fixed deposits with a 16% return are not tax-...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=uQc6Tib329A\n,"The financial influencer claims that Gropetal,...",The claim of a 16% tax-free return in one year...,<|prompter|>You are a Financial Contrarian Wri...,false,true
4,SQJxDy9F_7o,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,Amrev,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,सबसे पहले मेरा एक बहुत अफरेंट सबवाल है कि जो आ...,"First of all, my question is what suggestions ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAcHBw...,https://www.youtube.com/watch?v=SQJxDy9F_7o,The financial influencer suggests that young i...,The claim that businesses with less risk have ...,<|prompter|>You are a Financial Contrarian Wri...,true,neutral


## vLLM Inference Server Engine for increased inference throughput and latency

In [5]:
# downloads finetuned model from huggingface hub
finetuned_model = "skshreyas714/skeptic-justify-205"

llm = LLM(model=finetuned_model, tensor_parallel_size=1)

INFO 10-30 04:22:59 llm_engine.py:72] Initializing an LLM engine with config: model='skshreyas714/skeptic-justify-205', tokenizer='skshreyas714/skeptic-justify-205', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, seed=0)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 10-30 04:23:16 llm_engine.py:207] # GPU blocks: 1330, # CPU blocks: 2048


In [10]:
ids = [i for i in range(len(df))]
idx = random.choice(ids)
print(idx)

prompt = df.iloc[idx]["text"]
justify = df.iloc[idx]["Justification"]
label = df.iloc[idx]["predicted_label"]
actual = df.iloc[idx]["labels"]

print(f"Claim Summary: {prompt}")
print(f"Actual Output: {justify}")
print(f"Actual Label: {actual}")
print(f"Predicted Label: {label}")

13
Claim Summary: <|prompter|>You are a Financial Contrarian Writer. You are provided with a
Summary of Claims made by financial influencer and your task is to write
justifications for those claims. Claim Summary: The financial influencer claims that the top two penny stocks Rage Individual has invested in are DB Realty and Broadcast Limited, currently priced at 29.5 and 36 Indian Rupees respectively. He suggests that these stocks have a significant upside potential, with prices expected to reach around 42 to 45 Indian Rupees. However, he also warns that like any normal blue chip stock, investments in these penny stocks carry risks.</s><|assistant|>
Actual Output: The claim about the current prices of DB Realty and Broadcast Limited can be verified by checking the current market prices. The claim about the potential upside of these stocks is speculative and based on the influencer's personal analysis or opinion, which may or may not be accurate. The claim about the risks associated wit

In [7]:
test_prompts = df["text"].tolist()

In [15]:
sampling_params = SamplingParams(temperature=1.0, max_tokens=200, presence_penalty=1.5,
                                 frequency_penalty=1.5, top_p=0.9, top_k=50, best_of=10,
                                 skip_special_tokens=True, use_beam_search=False,
                                 early_stopping=False)

outputs = llm.generate(test_prompts, sampling_params)

predicted = []
for output in outputs:
    generated_text = output.outputs[0].text.strip(" ")
    predicted.append(generated_text)
    print(f"Generated text: {generated_text!r}")

Processed prompts: 100%|██████████| 21/21 [00:16<00:00,  1.30it/s]

Generated text: "The claims made by the influencer are true as they are based on the financial results of TCS. The increase in profits, revenues, and margins can be verified from the company's financial reports. However, the claim about constant currency growth being disappointing is subjective and may depend on individual analysis. The announcement of a dividend and buyback can be confirmed through official company announcements. It's important to note that past performance is not indicative of future results and investors should do their own research before making investment decisions."
Generated text: "The claim that the conflict could impact India's economy is plausible as it could lead to increases in petroleum prices and shipping costs, which would affect Indian companies. However, the claim about the India-Middle East economic corridor project is speculative and depends on the outcome of the conflict. The claim about stock market volatility is true as conflicts often lead to fin

In [12]:
response = {
    "claim": df.iloc[idx]["Summary_Claims"],
    "actual_justification": df.iloc[idx]["Justification"],
    "generated_justification": generated_text,
    "actual_label": actual, "predicted_label": label
           }

print(response)

{'claim': 'The financial influencer claims that the top two penny stocks Rage Individual has invested in are DB Realty and Broadcast Limited, currently priced at 29.5 and 36 Indian Rupees respectively. He suggests that these stocks have a significant upside potential, with prices expected to reach around 42 to 45 Indian Rupees. However, he also warns that like any normal blue chip stock, investments in these penny stocks carry risks.', 'actual_justification': "The claim about the current prices of DB Realty and Broadcast Limited can be verified by checking the current market prices. The claim about the potential upside of these stocks is speculative and based on the influencer's personal analysis or opinion, which may or may not be accurate. The claim about the risks associated with investing in penny stocks is generally true, as these stocks are often more volatile and less liquid than blue chip stocks. Therefore, while the specific price predictions may be questionable, the general a

### Evaluation with ROUGE, METEOR, BERT-Score metrics

In [13]:
def compute_metrics(generated, reference):
    rouge_scores = rouge.get_scores(generated, reference)
    meteor = meteor_score([reference.split()], generated.split())
    # bert_precision, bert_recall, bert_f1 = score([generated], [reference], lang="en")
    # bert_f1 = bert_f1.detach().numpy().tolist()[0]
    return {"rouge_scores": rouge_scores, "meteor": meteor}#, "bert_score": bert_f1}

In [16]:
r, m, b = [], [], []
for i in range(len(df)):
    metrics = compute_metrics(predicted[i], df.iloc[i]["Justification"])
    rouge_m, meteor_m = metrics["rouge_scores"][0]["rouge-l"]["f"], metrics["meteor"]#, metrics["bert_score"]
    r.append(rouge_m)
    m.append(meteor_m)
    # b.append(bert_m)

In [17]:
rouge_s = np.round(np.mean(r), 2)
metoer_s = np.round(np.mean(m), 2)
# bert_s = np.round(np.mean(b), 2)

In [18]:
print(f"Rouge Score for Test Set: {round(rouge_s,2)}\n")
print(f"Meteor Score for Test Set: {round(metoer_s,2)}\n")
# print(f"BERT Scores: {round(bert_s,2)*100}%\n")

Rouge Score for Test Set: 0.56

Meteor Score for Test Set: 0.43



### Dumping the generated justifications into test dataframe

In [19]:
df["Generated_Justifications"] = predicted

In [20]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,text,labels,predicted_label,Generated_Justifications
0,7BObqdlgd5A,TCS Q2 Results 2023-24 Highlights | TCS Share ...,5paisa,TCS has just announced its Quarterly Results. ...,"Hi guys, Quarter 2 FY24 result season ki shirv...","""Hi guys, the Q2 FY24 result season has begun ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQcICA...,https://www.youtube.com/watch?v=7BObqdlgd5A,The financial influencer claims that TCS's Q2 ...,The claims made by the influencer are true if ...,<|prompter|>You are a Financial Contrarian Wri...,true,true,The claims made by the influencer are true as ...
1,R_OryHP3Fcg,Israel-Hamas Conflict's Impact on India #shorts,5paisa,Gain insight into how the Israel-Hamas Conflic...,बिलियन डौलर का ट्रेटिंग रेलेशन्चिप खत्रे में ह...,The trading relationship between Israel and Ha...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=R_OryHP3Fcg,The financial influencer claims that the confl...,The claims made by the influencer are plausibl...,<|prompter|>You are a Financial Contrarian Wri...,false,true,The claim that the conflict could impact India...
2,qX3J9Tq0XYU,YOUTUBE SE INCOME || MY FIRST INCOME,Amrev,YOUTUBE SE INCOME || MY FIRST INCOME\r\n\r\n...,so yeah parents do say a lot but it's okay wh...,"""So yeah, parents do say a lot, but it's okay....",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=qX3J9Tq0XYU,No claims made,No claims to analyse,<|prompter|>You are a Financial Contrarian Wri...,neutral,neutral,No claims to analyse.
3,uQc6Tib329A,Growpital Review - 16% TAX FREE Return | Fixed...,Shrija Saha,Fixed Income - 16% Tax FREE Return - Growpita...,Fixed deposit with 16% returns वो भी tax-free ...,"""Fixed deposits with a 16% return are not tax-...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=uQc6Tib329A\n,"The financial influencer claims that Gropetal,...",The claim of a 16% tax-free return in one year...,<|prompter|>You are a Financial Contrarian Wri...,false,true,The claim that Gropetal offers a fixed return ...
4,SQJxDy9F_7o,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,Amrev,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,सबसे पहले मेरा एक बहुत अफरेंट सबवाल है कि जो आ...,"First of all, my question is what suggestions ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAcHBw...,https://www.youtube.com/watch?v=SQJxDy9F_7o,The financial influencer suggests that young i...,The claim that businesses with less risk have ...,<|prompter|>You are a Financial Contrarian Wri...,true,neutral,The claim that businesses with less risk have ...


In [21]:
df.drop(columns=["text"], inplace=True)

In [ ]:
df.to_csv("data/mistral-lite-finetuned-labels-justifications.csv", index=False)